# Race Time Predictor

Prédiction du temps de course à partir d'une trace GPX,
basée sur le modèle de coût métabolique de Minetti (2002).

**Modèles intégrés**
- Minetti (2002) : coût métabolique en fonction de la pente
- Fatigue exponentielle à plateau (calibrée sur historique athlète)
- Plafond vitesse en descente
- Dégradation thermique (Ely et al. 2007) avec correction altitudinale (Rolland 2003)


In [10]:
import sys
import warnings
import requests
warnings.filterwarnings('ignore')

sys.path.insert(0, '../core')

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gpx_race import parse_gpx, segment_trace
from race_predictor import (
    race_summary, format_time, format_pace,
    minetti_speed_ratio, minetti_cost, minetti_cost_flat
)


## 1. Paramètres

In [ ]:
# --- Trace GPX ---
GPX_PATH  = '/home/gsainton/Downloads/ultra-trail-du-haut-giffre-2026-trail-du-mont-blanc-des-dames.gpx'
RACE_NAME = 'UTHG TMBD 2026'
RACE_DATE = '2026-06-21'   # YYYY-MM-DD — utilisé pour la météo forecast

# --- Heure de départ ---
START_TIME = '05:00'  # HH:MM

# --- Vitesse de référence sur terrain plat ---
V_FLAT_KMH    = 8.5   # km/h
ATHLETE_FACTOR = 0.85  # < 1.0 = terrain technique / prudence

# --- Segmentation RDP ---
EPSILON_M  = 2.0   # tolérance altitude (m)
MIN_SEG_KM = 0.05  # longueur minimale d'un segment (km)

# --- Modèle de fatigue (calibré sur EcoTrail, Imperial, SainteLyon) ---
FATIGUE_A   = 0.808  # plateau asymptotique
FATIGUE_K   = 4.084  # vitesse de dégradation
USE_FATIGUE = True   # False = Minetti pur

# --- Plafond vitesse en descente ---
V_DOWNHILL_MAX_KMH = 10.0

# --- Modèle thermique (Ely et al. 2007 / Rolland 2003) ---
T_THRESHOLD     = 15.0   # °C — seuil de dégradation
ALPHA_HEAT      = 0.004  # 0.4 % de dégradation par degré au-dessus du seuil
TEMP_LAPSE_RATE = 0.0065 # °C/m — gradient altitudinal standard
USE_THERMAL     = True   # False = sans effet thermique

# --- Ravitaillements : (nom, distance km) ---
RAVITOS = [
    ('Joux Plane',    18.3),
    ('Golèse',        30.0),
    ('Refuge Bostan', 38.1),
    ('Vallon',        46.6),
    ('Sixt',          59.2),
    ('Arrivée',       71.3),
]

# --- Barrières horaires : (nom, distance km, heure HH:MM ou None) ---
BARRIERES = [
    ('Joux Plane',    18.3, '9:00'),
    ('Golèse',        30.0, None),
    ('Refuge Bostan', 38.1, None),
    ('Vallon',        46.6, '17:00'),
    ('Sixt',          59.2, '21:00'),
    ('Arrivée',       71.3, '23:59'),
]


## 2. Fonctions utilitaires

In [12]:
def parse_hhmm(hhmm):
    """Convert HH:MM string to minutes since midnight."""
    h, m = map(int, hhmm.split(':'))
    return h * 60 + m


def plateau_fatigue(x, a, k):
    """
    Exponential fatigue model with plateau.

    x : fraction de distance parcourue [0, 1]
    a : facteur asymptotique (plateau en fin de course)
    k : vitesse de dégradation initiale
    """
    return a + (1 - a) * np.exp(-k * x)


def interpolate_temp_at_time(df_weather, time_minutes_since_midnight):
    """Interpolate temperature (°C) from hourly forecast at a given time."""
    hours = df_weather['time'].dt.hour + df_weather['time'].dt.minute / 60
    temp  = df_weather['temperature_2m'].values
    return float(np.interp(time_minutes_since_midnight / 60, hours, temp))


def altitudinal_correction(temp_ref, alt_ref, alt_target,
                            lapse_rate=TEMP_LAPSE_RATE):
    """Apply standard lapse rate correction to temperature."""
    return temp_ref - lapse_rate * (alt_target - alt_ref)


def thermal_factor(temp_c, t_threshold=T_THRESHOLD, alpha=ALPHA_HEAT):
    """Speed degradation factor due to heat (Ely et al. 2007)."""
    return max(0.0, 1.0 - alpha * max(0.0, temp_c - t_threshold))


def predict_segments_full(df_segments, v_flat_kmh, athlete_factor,
                          fatigue_a, fatigue_k, use_fatigue,
                          v_downhill_max_kmh, start_min,
                          df_weather, alt_ref, use_thermal):
    """
    Predict speed and time per segment.

    Combines Minetti cost model with fatigue, downhill cap, and thermal effect.
    """
    total_dist = df_segments['length_km'].sum()
    rows  = []
    t_cum = 0.0
    d_cum = 0.0

    for _, seg in df_segments.iterrows():
        dist_pct = (d_cum + seg['length_km'] / 2) / total_dist
        alt_mid  = (seg['ele_start'] + seg['ele_end']) / 2

        fatigue = plateau_fatigue(dist_pct, fatigue_a, fatigue_k) if use_fatigue else 1.0

        heat, temp_seg = 1.0, None
        if use_thermal and df_weather is not None:
            temp_ref = interpolate_temp_at_time(df_weather, start_min + t_cum)
            temp_seg = altitudinal_correction(temp_ref, alt_ref, alt_mid)
            heat     = thermal_factor(temp_seg)

        ratio  = minetti_speed_ratio(seg['slope_mean_pct'])
        v_pred = v_flat_kmh * ratio * athlete_factor * fatigue * heat

        if seg['slope_mean_pct'] < -3:
            v_pred = min(v_pred, v_downhill_max_kmh)
        v_pred = max(v_pred, 1.0)

        pace  = 60.0 / v_pred
        t_min = seg['length_km'] / v_pred * 60.0
        t_cum += t_min
        d_cum += seg['length_km']

        rows.append({
            **seg.to_dict(),
            'speed_kmh':       round(v_pred, 2),
            'pace_min_per_km': round(pace, 2),
            'time_min':        round(t_min, 1),
            'time_cum_min':    round(t_cum, 1),
            'effort_index':    round(minetti_cost(seg['slope_mean_pct']) / minetti_cost_flat(), 2),
            'fatigue_factor':  round(fatigue, 3),
            'temp_seg_c':      round(temp_seg, 1) if temp_seg is not None else None,
            'thermal_factor':  round(heat, 3),
        })

    return pd.DataFrame(rows)


## 3. Météo forecast

In [13]:
_FORECAST_URL = 'https://api.open-meteo.com/v1/forecast'
_HOURLY_VARS  = [
    'temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
    'precipitation', 'wind_speed_10m', 'shortwave_radiation',
]


def fetch_weather_forecast(lat, lon, date_str, timezone='Europe/Paris', date_end=None):
    """
    Fetch hourly weather forecast from Open-Meteo (up to 16 days ahead).

    Returns pd.DataFrame or None on error.
    """
    params = {
        'latitude':        lat,
        'longitude':       lon,
        'start_date':      date_str,
        'end_date':        date_end or date_str,
        'hourly':          ','.join(_HOURLY_VARS),
        'wind_speed_unit': 'kmh',
        'timezone':        timezone,
    }
    try:
        resp = requests.get(_FORECAST_URL, params=params, timeout=10)
        resp.raise_for_status()
        hourly = resp.json().get('hourly', {})
    except Exception as e:
        print(f'Erreur Open-Meteo forecast : {e}')
        return None

    df = pd.DataFrame({
        'time':                 pd.to_datetime(hourly['time']),
        'temperature_2m':       hourly['temperature_2m'],
        'relative_humidity_2m': hourly['relative_humidity_2m'],
        'apparent_temperature': hourly['apparent_temperature'],
        'precipitation':        hourly['precipitation'],
        'wind_speed_10m':       hourly['wind_speed_10m'],
        'shortwave_radiation':  hourly['shortwave_radiation'],
    })
    df = df[df['time'].dt.date == pd.to_datetime(date_str).date()].reset_index(drop=True)

    # WBGT simplifié (Bernard & Kenney 1994)
    df['wbgt'] = (
        0.567 * df['temperature_2m']
        + 0.393 * (df['relative_humidity_2m'] / 100 * 6.105
                   * np.exp(17.27 * df['temperature_2m'] / (237.7 + df['temperature_2m'])))
        + 3.94
    )
    return df


# Centroïde de la trace
df_gpx   = parse_gpx(GPX_PATH)
df_seg   = segment_trace(df_gpx, epsilon_m=EPSILON_M, min_seg_km=MIN_SEG_KM)
start_min = parse_hhmm(START_TIME)
lat_c    = df_gpx['lat'].mean()
lon_c    = df_gpx['lon'].mean()
alt_ref  = float(df_gpx['ele'].mean())

df_weather = fetch_weather_forecast(lat_c, lon_c, RACE_DATE)
if df_weather is not None:
    print(f'Prévisions pour le {RACE_DATE} — centroïde ({lat_c:.3f}N, {lon_c:.3f}E)')
    print(df_weather[['time', 'temperature_2m', 'relative_humidity_2m', 'wbgt']].to_string(index=False))
else:
    print('Prévision indisponible — effet thermique désactivé.')
    USE_THERMAL = False


Prévisions pour le 2026-06-21 — centroïde (46.104N, 6.739E)
               time  temperature_2m  relative_humidity_2m      wbgt
2026-06-21 00:00:00            18.9                    66 20.306423
2026-06-21 01:00:00            18.5                    67 20.023743
2026-06-21 02:00:00            18.2                    67 19.749540
2026-06-21 03:00:00            17.8                    67 19.386576
2026-06-21 04:00:00            17.5                    66 19.037888
2026-06-21 05:00:00            17.7                    64 19.058074
2026-06-21 06:00:00            18.7                    61 19.700223
2026-06-21 07:00:00            20.3                    57 20.772289
2026-06-21 08:00:00            22.0                    53 21.905931
2026-06-21 09:00:00            23.6                    50 23.028813
2026-06-21 10:00:00            25.2                    47 24.131852
2026-06-21 11:00:00            26.5                    45 25.069133
2026-06-21 12:00:00            27.2                    4

## 4. Prédiction par segment

In [14]:
df_pred = predict_segments_full(
    df_seg,
    v_flat_kmh=V_FLAT_KMH,
    athlete_factor=ATHLETE_FACTOR,
    fatigue_a=FATIGUE_A,
    fatigue_k=FATIGUE_K,
    use_fatigue=USE_FATIGUE,
    v_downhill_max_kmh=V_DOWNHILL_MAX_KMH,
    start_min=start_min,
    df_weather=df_weather,
    alt_ref=alt_ref,
    use_thermal=USE_THERMAL,
)

summary = race_summary(df_pred, race_name=RACE_NAME)
end_min = start_min + summary['total_min']

print(f"Course   : {summary['race_name']}")
print(f"Distance : {summary['distance_km']} km  |  D+ : {summary['dplus_m']} m  |  D- : {summary['dmoins_m']} m")
print(f"Segments : {summary['n_segments']}")
print(f"Temps    : {summary['total_time_str']}  (allure moy. {summary['avg_pace_str']} /km)")
print(f"Arrivée  : {int(end_min)//60 % 24:02d}h{int(end_min)%60:02d}")


Course   : Trail du Mont-Blanc des Dames 2026
Distance : 71.28 km  |  D+ : 4840 m  |  D- : 4855 m
Segments : 20
Temps    : 14h 47min  (allure moy. 12'27" /km)
Arrivée  : 19h47


In [15]:
def plot_temperature_forecast(df_weather, start_min, total_min, race_name):
    """Plot hourly temperature forecast with race window highlighted."""
    if df_weather is None:
        print("Pas de données météo disponibles.")
        return

    hours = df_weather['time'].dt.hour + df_weather['time'].dt.minute / 60
    temp  = df_weather['temperature_2m'].values
    wbgt  = df_weather['wbgt'].values

    # Fenêtre de course
    t_start_h = start_min / 60
    t_end_h   = (start_min + total_min) / 60

    fig = go.Figure()

    # Zone de course
    fig.add_vrect(
        x0=t_start_h, x1=t_end_h,
        fillcolor='rgba(52,152,219,0.12)',
        line_width=0,
        annotation_text='Fenêtre de course',
        annotation_position='top left',
        annotation_font=dict(color='#3498db', size=10),
    )

    fig.add_trace(go.Scatter(
        x=hours, y=temp,
        mode='lines+markers',
        name='Température 2m (°C)',
        line=dict(color='#e67e22', width=2.5),
        marker=dict(size=5),
        hovertemplate='%{x:.1f}h : %{y:.1f}°C<extra></extra>',
    ))

    fig.add_trace(go.Scatter(
        x=hours, y=wbgt,
        mode='lines',
        name='WBGT (°C)',
        line=dict(color='#e74c3c', width=1.5, dash='dot'),
        hovertemplate='%{x:.1f}h : WBGT=%{y:.1f}°C<extra></extra>',
    ))

    # Seuil thermique
    fig.add_hline(
        y=T_THRESHOLD,
        line=dict(color='#f1c40f', width=1, dash='dash'),
        annotation_text=f'Seuil dégradation ({T_THRESHOLD}°C)',
        annotation_position='right',
        annotation_font=dict(color='#f1c40f', size=9),
    )

    fig.update_layout(
        template='plotly_dark',
        title=f'Prévision météo — {race_name}',
        xaxis=dict(
            title='Heure de la journée',
            tickvals=list(range(0, 25, 2)),
            ticktext=[f'{h:02d}h00' for h in range(0, 25, 2)],
        ),
        yaxis=dict(title='Température (°C)'),
        height=380,
        margin=dict(t=60, b=50, l=60, r=20),
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.show()


plot_temperature_forecast(df_weather, start_min, summary['total_min'], RACE_NAME)


## 5. Profil altimétrique

In [ ]:
SLOPE_COLORS = {
    'montée raide':   '#e74c3c',
    'montée':         '#e67e22',
    'plat':           '#2ecc71',
    'descente':       '#3498db',
    'descente raide': '#9b59b6',
}


def make_elevation_profile(df_gpx, df_pred, ravitos, barrieres, start_min, df_weather=None):
    """Build interactive elevation profile with segments, ravitos, cutoffs, and optional temperature."""
    fig     = go.Figure()
    dist_km = df_gpx['dist_cum'].values / 1000.0
    ele     = df_gpx['ele'].values

    fig.add_trace(go.Scatter(
        x=dist_km, y=ele,
        fill='tozeroy', fillcolor='rgba(150,150,150,0.15)',
        line=dict(color='rgba(0,0,0,0)', width=0),
        showlegend=False, hoverinfo='skip'
    ))

    legend_added = set()
    for _, seg in df_pred.iterrows():
        cat   = seg['slope_category']
        color = SLOPE_COLORS.get(cat, '#888')
        mask  = (dist_km >= seg['dist_start_km']) & (dist_km <= seg['dist_end_km'])
        show_legend = cat not in legend_added
        legend_added.add(cat)

        factors = f"fatigue={seg['fatigue_factor']:.2f}"
        if seg['temp_seg_c'] is not None:
            factors += f" | T={seg['temp_seg_c']:.1f}°C therm={seg['thermal_factor']:.2f}"

        fig.add_trace(go.Scatter(
            x=dist_km[mask], y=ele[mask],
            mode='lines',
            line=dict(color=color, width=2.5),
            name=cat, legendgroup=cat, showlegend=show_legend,
            hovertemplate=(
                f"<b>{cat}</b><br>"
                "Dist : %{x:.1f} km | Alt : %{y:.0f} m<br>"
                f"{factors}<extra></extra>"
            )
        ))

    for nom, dist in ravitos:
        t_min = float(df_pred[df_pred['dist_end_km'] <= dist]['time_min'].sum())
        heure = (start_min + t_min) % (24 * 60)
        heure_str = f"{int(heure)//60:02d}h{int(heure)%60:02d}"
        fig.add_vline(x=dist, line=dict(color='#f1c40f', width=1.5, dash='dot'))
        fig.add_annotation(
            x=dist, y=max(ele) * 1.02,
            text=f"<b>{nom}</b><br>{heure_str}",
            showarrow=False, font=dict(size=10, color='#f1c40f'),
            bgcolor='rgba(0,0,0,0.5)', bordercolor='#f1c40f', borderwidth=1,
            xanchor='center'
        )

    for barriere in barrieres:
        if barriere[2] is None:
            continue
        nom, dist, heure_limite = barriere
        marge_min = parse_hhmm(heure_limite) - (
            start_min + float(df_pred[df_pred['dist_end_km'] <= dist]['time_min'].sum())
        )
        signe = '+' if marge_min >= 0 else ''
        fig.add_vline(x=dist, line=dict(color='#e74c3c', width=1.5, dash='dash'))
        fig.add_annotation(
            x=dist, y=min(ele) * 0.85,
            text=f"<b>Barr. {heure_limite}</b><br>{signe}{int(marge_min)}min",
            showarrow=False, font=dict(size=9, color='#e74c3c'),
            bgcolor='rgba(0,0,0,0.5)', bordercolor='#e74c3c', borderwidth=1,
            xanchor='center'
        )

    # Température corrigée de l'altitude par segment
    if df_weather is not None and 'temp_seg_c' in df_pred.columns:
        seg_dists = (df_pred['dist_start_km'] + df_pred['dist_end_km']) / 2
        seg_temps = df_pred['temp_seg_c'].values
        #wbgt  = df_pred['wbgt'].values

        fig.add_trace(go.Scatter(
            x=seg_dists,
            y=seg_temps,
            mode='lines+markers',
            name='T° corrigée altitude (°C)',
            yaxis='y2',
            line=dict(color='#f39c12', width=2, dash='dot'),
            marker=dict(size=6, symbol='diamond'),
            hovertemplate='%{x:.1f} km : %{y:.1f}°C<extra>T°</extra>',
        ))

        # Seuil thermique sur axe y2
        fig.add_hline(
            y=T_THRESHOLD, line=dict(color='#f1c40f', width=1, dash='dash'),
            annotation_text=f'Seuil {T_THRESHOLD}°C',
            annotation_position='top right',
            annotation_font=dict(color='#f1c40f', size=9),
            yref='y2',
        )

    fig.update_layout(
        template='plotly_dark',
        title=dict(text=f'Profil alti — {RACE_NAME}', font=dict(size=16)),
        xaxis=dict(title='Distance (km)', showgrid=True, gridcolor='rgba(255,255,255,0.1)'),
        yaxis=dict(title='Altitude (m)', showgrid=True, gridcolor='rgba(255,255,255,0.1)'),
        yaxis2=dict(
            title='Température (°C)',
            overlaying='y',
            side='right',
            showgrid=False,
            tickfont=dict(color='#f39c12'),
            title_font=dict(color='#f39c12'),
        ),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        height=520, margin=dict(t=80, b=50, l=60, r=60),
    )
    return fig

make_elevation_profile(df_gpx, df_pred, RAVITOS, BARRIERES, start_min, df_weather).show()

KeyError: 'wbgt'

## 6. Table des segments

In [17]:
def style_segments(df):
    """Apply color styling to the segments table."""
    def color_slope(val):
        return {
            'montée raide':   'color: #e74c3c',
            'montée':         'color: #e67e22',
            'plat':           'color: #2ecc71',
            'descente':       'color: #3498db',
            'descente raide': 'color: #9b59b6',
        }.get(val, '')

    return (
        df.style
        .applymap(color_slope, subset=['Catégorie'])
        .set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th',
                            'props': [('text-align', 'center'), ('font-weight', 'bold')]}])
    )


t_cum  = start_min
heures = []
for _, seg in df_pred.iterrows():
    t_cum += seg['time_min']
    h = t_cum % (24 * 60)
    heures.append(f"{int(h)//60:02d}h{int(h)%60:02d}")

cols_display = [
    'dist_start_km', 'dist_end_km', 'length_km',
    'dplus_m', 'dmoins_m', 'slope_mean_pct', 'slope_category',
    'speed_kmh', 'pace_min_per_km', 'time_min',
    'effort_index', 'fatigue_factor', 'temp_seg_c', 'thermal_factor',
]
df_display = df_pred[cols_display].copy()
df_display['pace_min_per_km'] = df_display['pace_min_per_km'].apply(format_pace)
df_display['time_min']        = df_display['time_min'].apply(format_time)
df_display['heure_arr']       = heures
df_display = df_display.rename(columns={
    'dist_start_km':  'De (km)',
    'dist_end_km':    'À (km)',
    'length_km':      'Dist (km)',
    'dplus_m':        'D+ (m)',
    'dmoins_m':       'D- (m)',
    'slope_mean_pct': 'Pente (%)',
    'slope_category': 'Catégorie',
    'speed_kmh':      'Vitesse (km/h)',
    'pace_min_per_km':'Allure (/km)',
    'time_min':       'Temps',
    'effort_index':   'Effort',
    'fatigue_factor': 'Fatigue',
    'temp_seg_c':     'T° (°C)',
    'thermal_factor': 'Therm.',
    'heure_arr':      'Heure arr.',
})

style_segments(df_display)


,De (km),À (km),Dist (km),D+ (m),D- (m),Pente (%),Catégorie,Vitesse (km/h),Allure (/km),Temps,Effort,Fatigue,T° (°C),Therm.,Heure arr.
0,0.000000,3.669185,3.669185,120.188624,15.498856,2.853216,plat,5.910000,"10'09""",37min,1.160000,0.981000,22.000000,0.972000,05h37
1,3.669185,6.701304,3.032119,407.312847,0.000000,13.433273,montée raide,3.470000,"17'16""",52min,1.930000,0.951000,21.000000,0.976000,06h29
2,6.701304,9.165880,2.464576,11.287022,178.272638,-6.775429,descente,9.410000,"6'22""",15min,0.700000,0.930000,21.400000,0.975000,06h45
3,9.165880,13.084866,3.918986,593.663289,0.516509,15.135210,montée raide,3.100000,"19'19""",1h 15min,2.070000,0.910000,20.400000,0.978000,08h01
4,13.084866,14.366802,1.281936,9.354037,78.373325,-5.383989,descente,8.450000,"7'05""",9min,0.750000,0.895000,20.800000,0.977000,08h10
5,14.366802,15.508542,1.141740,51.440667,0.207911,4.487252,montée,4.950000,"12'07""",13min,1.270000,0.890000,21.100000,0.976000,08h24
6,15.508542,16.363663,0.855120,197.311540,0.000000,23.074123,montée raide,2.240000,"26'48""",22min,2.790000,0.885000,20.700000,0.977000,08h47
7,16.363663,18.345949,1.982287,38.566525,172.124551,-6.737574,descente,8.890000,"6'45""",13min,0.700000,0.879000,21.100000,0.976000,09h00
8,18.345949,19.918321,1.572372,314.411572,0.000000,19.996006,montée raide,2.460000,"24'23""",38min,2.500000,0.872000,20.900000,0.977000,09h38
9,19.918321,26.620688,6.702367,153.818568,981.034652,-12.342149,descente raide,10.000000,"6'00""",40min,0.550000,0.859000,23.500000,0.966000,10h19


## 7. Décomposition par tronçons

In [18]:
def build_splits_table(df_pred, ravitos, barrieres, start_min):
    """Build a splits table grouped by ravito sections."""
    bounds = [0.0] + [d for _, d in ravitos]
    names  = ['Départ'] + [n for n, _ in ravitos]
    barrieres_dict = {d: (n, h) for n, d, h in barrieres if h is not None}

    rows  = []
    t_cum = start_min

    for k in range(len(bounds) - 1):
        d_start = bounds[k]
        d_end   = bounds[k + 1]
        label   = f"{names[k]} → {names[k+1]}"

        mask = (
            (df_pred['dist_start_km'] < d_end + 0.01) &
            (df_pred['dist_end_km']   > d_start - 0.01)
        )
        sub = df_pred[mask]
        if sub.empty:
            continue

        t_seg = dp = dm = dist = 0.0
        for _, seg in sub.iterrows():
            frac   = (min(seg['dist_end_km'], d_end) - max(seg['dist_start_km'], d_start)) / seg['length_km']
            t_seg += seg['time_min'] * frac
            dp    += seg['dplus_m']  * frac
            dm    += seg['dmoins_m'] * frac
            dist  += seg['length_km'] * frac

        speed = dist / (t_seg / 60) if t_seg > 0 else 0
        pace  = 60 / speed if speed > 0 else None

        t_cum    += t_seg
        heure_arr = t_cum % (24 * 60)
        heure_str = f"{int(heure_arr)//60:02d}h{int(heure_arr)%60:02d}"

        barriere_str = ''
        if d_end in barrieres_dict:
            _, h_lim = barrieres_dict[d_end]
            marge    = parse_hhmm(h_lim) - t_cum
            signe    = '+' if marge >= 0 else ''
            barriere_str = f"{h_lim} ({signe}{int(marge)}min)"

        rows.append({
            'Tronçon':        label,
            'Dist (km)':      round(float(dist), 1),
            'D+ (m)':         int(dp),
            'D- (m)':         int(dm),
            'Temps':          format_time(t_seg),
            'Vitesse (km/h)': round(float(speed), 1),
            'Allure (/km)':   format_pace(pace),
            'Heure arr.':     heure_str,
            'Barrière':       barriere_str,
        })

    return pd.DataFrame(rows)


def style_splits(df):
    """Apply color styling to the splits table."""
    def color_barriere(val):
        if not val:
            return ''
        return 'color: #2ecc71; font-weight: bold' if '+' in val else 'color: #e74c3c; font-weight: bold'

    return (
        df.style
        .applymap(color_barriere, subset=['Barrière'])
        .set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th',
                            'props': [('text-align', 'center'), ('font-weight', 'bold')]}])
    )


df_splits = build_splits_table(df_pred, RAVITOS, BARRIERES, start_min)
print(f"Départ : {START_TIME} | V_flat : {V_FLAT_KMH} km/h | Facteur athlète : {ATHLETE_FACTOR}")
print(f"Fatigue : a={FATIGUE_A} k={FATIGUE_K} | Descente max : {V_DOWNHILL_MAX_KMH} km/h | Thermique : {USE_THERMAL}")
print()
style_splits(df_splits)


Départ : 05:00 | V_flat : 8.5 km/h | Facteur athlète : 0.85
Fatigue : a=0.808 k=4.084 | Descente max : 10.0 km/h | Thermique : True



,Tronçon,Dist (km),D+ (m),D- (m),Temps,Vitesse (km/h),Allure (/km),Heure arr.,Barrière
0,Départ → Joux Plane,18.300000,1428,441,4h 00min,4.600000,"13'07""",09h00,9:00 (0min)
1,Joux Plane → Golèse,11.700000,1073,985,2h 38min,4.400000,"13'32""",11h38,
2,Golèse → Refuge Bostan,8.100000,652,747,1h 49min,4.400000,"13'29""",13h27,
3,Refuge Bostan → Vallon,8.500000,159,1022,1h 08min,7.400000,"8'05""",14h36,17:00 (+143min)
4,Vallon → Sixt,12.600000,782,841,2h 38min,4.800000,"12'36""",17h15,21:00 (+224min)
5,Sixt → Arrivée,12.100000,743,817,2h 31min,4.800000,"12'34""",19h47,23:59 (+251min)


## 8. Heures de passage vs barrières

In [19]:
def plot_passage_times(df_splits, barrieres, start_min):
    """Bar chart of predicted passage times vs cutoff times."""
    labels    = [r.split('→')[-1].strip() for r in df_splits['Tronçon']]
    t_passage = []
    for _, row in df_splits.iterrows():
        h_str = row['Heure arr.']
        t_passage.append(int(h_str[:2]) * 60 + int(h_str[3:]))

    labels_barr, t_barrieres = [], []
    for nom, dist, heure in barrieres:
        if heure is None:
            continue
        labels_barr.append(nom)
        t_barrieres.append(parse_hhmm(heure))

    def fmt(minutes):
        return f"{int(minutes)//60:02d}h{int(minutes)%60:02d}"

    tick_vals = list(range(int(min(t_passage + t_barrieres)) - 30,
                           int(max(t_passage + t_barrieres)) + 60, 60))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=labels, y=t_passage,
        name='Heure de passage prédite', marker_color='#3498db',
        text=[fmt(t) for t in t_passage], textposition='outside',
    ))
    fig.add_trace(go.Scatter(
        x=labels_barr, y=t_barrieres,
        mode='markers+lines+text', name='Barrière horaire',
        marker=dict(color='#e74c3c', size=10, symbol='diamond'),
        line=dict(color='#e74c3c', dash='dash'),
        text=[fmt(t) for t in t_barrieres], textposition='top center',
    ))
    fig.update_layout(
        template='plotly_dark',
        title='Heures de passage prédites vs barrières horaires',
        xaxis=dict(title='Point de passage'),
        yaxis=dict(title='Heure', tickvals=tick_vals, ticktext=[fmt(v) for v in tick_vals]),
        height=450, margin=dict(t=60, b=50),
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    return fig


plot_passage_times(df_splits, BARRIERES, start_min).show()


## 9. Résumé global

In [20]:
end_min  = start_min + summary['total_min']
end_hhmm = f"{int(end_min)//60 % 24:02d}h{int(end_min)%60:02d}"

print('=' * 55)
print(f"  {summary['race_name']}")
print('=' * 55)
print(f"  Distance    : {summary['distance_km']} km")
print(f"  D+          : {summary['dplus_m']} m")
print(f"  D-          : {summary['dmoins_m']} m")
print(f"  Départ      : {START_TIME}")
print(f"  V_flat      : {V_FLAT_KMH} km/h  (facteur athlète {ATHLETE_FACTOR})")
print(f"  Fatigue     : a={FATIGUE_A}  k={FATIGUE_K}  actif={USE_FATIGUE}")
print(f"  Thermique   : seuil={T_THRESHOLD}°C  alpha={ALPHA_HEAT}  actif={USE_THERMAL}")
print(f"  Descente max: {V_DOWNHILL_MAX_KMH} km/h")
print(f"  Temps total : {summary['total_time_str']}")
print(f"  Allure moy. : {summary['avg_pace_str']} /km")
print(f"  Arrivée     : ~{end_hhmm}")
print('=' * 55)


  Trail du Mont-Blanc des Dames 2026
  Distance    : 71.28 km
  D+          : 4840 m
  D-          : 4855 m
  Départ      : 05:00
  V_flat      : 8.5 km/h  (facteur athlète 0.85)
  Fatigue     : a=0.808  k=4.084  actif=True
  Thermique   : seuil=15.0°C  alpha=0.004  actif=True
  Descente max: 10.0 km/h
  Temps total : 14h 47min
  Allure moy. : 12'27" /km
  Arrivée     : ~19h47
